In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, cross_val_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
!pip install -q dagshub mlflow

import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = user_secrets.get_secret("DAGSHUB_USERNAME")

REPO_NAME = "ML_assignment1"

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

dagshub.init(
    repo_name=REPO_NAME,
    repo_owner=DAGSHUB_USERNAME,
    mlflow=True
)

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow")

print("Connected")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 78.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=ee332d4f-6c46-4b5d-847e-2614c9f2f50d&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=23079f69f28c910e5da37b20baf7e80dc5a6c0705d1516d995523be1326f78fa




Accessing as njvar23

Initialized MLflow to track repo "njvar23/ML_assignment1"

Repository njvar23/ML_assignment1 initialized!

Connected


In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
df_exp1 = train_df.copy()
df_exp2 = train_df.copy()

In [4]:
train_df.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

# 1 Baseline: Keeping All Columns

In [5]:
with mlflow.start_run(run_name="Keep_All_Columns"):
    if 'Id' in df_exp1.columns:
        df_exp1 = df_exp1.drop(columns=['Id'])
    
    X1 = df_exp1.drop(columns=['SalePrice'])
    y1 = df_exp1['SalePrice']
    X_train1, X_val1, y_train1, y_val1 = train_test_split(X1, y1, test_size=0.2, random_state=42)

    cat_cols1 = X_train1.select_dtypes(include=['object']).columns
    num_cols1 = X_train1.select_dtypes(exclude=['object']).columns
    
    X_train1[cat_cols1] = X_train1[cat_cols1].fillna("None")
    X_val1[cat_cols1] = X_val1[cat_cols1].fillna("None")
    X_train1[num_cols1] = X_train1[num_cols1].fillna(X_train1[num_cols1].median())
    X_val1[num_cols1] = X_val1[num_cols1].fillna(X_train1[num_cols1].median())

    X_train_enc1 = pd.get_dummies(X_train1)
    X_val_enc1 = pd.get_dummies(X_val1)
    X_train_enc1, X_val_enc1 = X_train_enc1.align(X_val_enc1, join='left', axis=1, fill_value=0)

    lasso1 = Lasso(alpha=100, random_state=42, max_iter=10000)
    cv_scores1 = cross_val_score(lasso1, X_train_enc1, y_train1, cv=5, scoring='neg_mean_squared_error')
    rmse_avg1 = np.sqrt(-cv_scores1).mean()
    
    mlflow.log_param("nan_threshold", 1.0) 
    mlflow.log_param("alpha", 100)
    mlflow.log_metric("cv_rmse", rmse_avg1)
    
    print(f"Exp 1 RMSE (Keep All): {rmse_avg1:.2f}")
    print(f"Features after Encoding: {X_train_enc1.shape[1]}")

Exp 1 RMSE (Keep All): 33247.10
Features after Encoding: 301
🏃 View run Keep_All_Columns at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/07bd7c52f6334849a8a44ccf2cabaea0
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0


# 2 Dropping columns & Lasso Regression

In [6]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score
with mlflow.start_run(run_name="Dropping_40%"):
    missing_pr = df_exp2.isnull().sum() / len(df_exp2)
    cols_to_drop = missing_pr[missing_pr > 0.4].index.tolist()
    if 'Id' in df_exp2.columns:
        cols_to_drop.append('Id')
    df_exp2 = df_exp2.drop(columns = cols_to_drop)
    
    mlflow.log_param("nan_threshold", 0.4)
    mlflow.log_param("dropped columns", cols_to_drop)

    X = df_exp2.drop(columns=['SalePrice'])
    y = df_exp2['SalePrice']
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    cat_cols = X_train.select_dtypes(include=['object']).columns
    num_cols = X_train.select_dtypes(exclude=['object']).columns
    
    X_train[cat_cols] = X_train[cat_cols].fillna("None")
    X_val[cat_cols] = X_val[cat_cols].fillna("None")
    
    train_medians = X_train[num_cols].median()
    X_train[num_cols] = X_train[num_cols].fillna(train_medians)
    X_val[num_cols] = X_val[num_cols].fillna(train_medians)

    X_train_enc = pd.get_dummies(X_train)
    X_val_enc = pd.get_dummies(X_val)
    X_train_enc, X_val_enc = X_train_enc.align(X_val_enc, join='left', axis=1, fill_value=0)

    mlflow.log_param("impute_strategy", "median_and_none")
    mlflow.log_param("total_features_after_encoding", X_train_enc.shape[1])

    lasso_exp2 = Lasso(alpha=100, random_state=42, max_iter=10000)
    cv_scores_exp2 = cross_val_score(lasso_exp2, X_train_enc, y_train, cv=5, scoring='neg_mean_squared_error')
    rmse_exp2 = np.sqrt(-cv_scores_exp2).mean()
    
    lasso_exp2.fit(X_train_enc, y_train)
    
    mlflow.log_metric("cv_rmse", rmse_exp2)
    
    print(f"Exp 2 RMSE (Dropped 40%): {rmse_exp2:.2f}")
    print(f"Features: {X_train_enc.shape[1]}")

Exp 2 RMSE (Dropped 40%): 32400.80
Features: 274
🏃 View run Dropping_40% at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/b4fabbab697849d9a9603eb4840a8a9e
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0


# 3. Feature Selection (RFE)

In [7]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="RFE_Selection"):
    selector = RFE(estimator=LinearRegression(), n_features_to_select=50)
    selector.fit(X_train_enc, y_train)

    X_train_rfe = selector.transform(X_train_enc)
    X_val_rfe = selector.transform(X_val_enc)

    mlflow.log_param("selection_method", "RFE")
    mlflow.log_param("features_retained", 50)
    
    selected_features = X_train_enc.columns[selector.support_].tolist()
    print(f"RFE selected {len(selected_features)} features.")

RFE selected 50 features.
🏃 View run RFE_Selection at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/7bd7a9d0b1134eea8c45eaefbfac019b
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0


# 4. Training: Lasso Regression

In [8]:
with mlflow.start_run(run_name="Lasso_Regression_RFE"):
    alpha_val = 100
    lasso_1 = Lasso(alpha=alpha_val, random_state=42, max_iter=10000)
    cv_scores = cross_val_score(lasso_1, X_train_rfe, y_train, cv=5, scoring='neg_mean_squared_error')
    rmse_avg_1 = np.sqrt(-cv_scores).mean()
    lasso_1.fit(X_train_rfe, y_train)
    important_features = np.sum(lasso_1.coef_ != 0)
    
    mlflow.log_param("alpha", alpha_val)
    mlflow.log_metric("cv_rmse", rmse_avg_1)
    mlflow.log_metric("features_selected", important_features)
    
    print(f"Lasso RMSE (with RFE): {rmse_avg_1:.2f}")
    print(f"Lasso prioritized {important_features} features out of the 50 provided by RFE.")

Lasso RMSE (with RFE): 35079.03
Lasso prioritized 29 features out of the 50 provided by RFE.
🏃 View run Lasso_Regression_RFE at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/4352aa1e5fb74ffb9ce69b8ce18cec49
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0


## 4.1 Hyperparameter Tuning : Lower Alpha

In [9]:
with mlflow.start_run(run_name="Lasso_Alpha_Tuning"):
    alpha_val_2 = 10 
    lasso_2 = Lasso(alpha=alpha_val_2, random_state=42, max_iter=10000)
    cv_scores_2 = cross_val_score(lasso_2, X_train_rfe, y_train, cv=5, scoring='neg_mean_squared_error')
    rmse_avg_2 = np.sqrt(-cv_scores_2).mean()
    
    lasso_2.fit(X_train_rfe, y_train)
    important_features_2 = np.sum(lasso_2.coef_ != 0)
    
    mlflow.log_param("alpha", alpha_val_2)
    mlflow.log_param("approach", "Lower Alpha / Less Strict Selection")
    mlflow.log_metric("cv_rmse", rmse_avg_2)
    mlflow.log_metric("features_selected", important_features_2)
    
    print(f"Lasso Alpha 10 RMSE: {rmse_avg_2:.2f}")
    print(f"Lasso kept {important_features_2} out of the 50 RFE features.")
    
    improvement = rmse_avg_1 - rmse_avg_2
    print(f"Improvement over alpha=100: {improvement:.2f}")

Lasso Alpha 10 RMSE: 35874.94
Lasso kept 50 out of the 50 RFE features.
Improvement over alpha=100: -795.92
🏃 View run Lasso_Alpha_Tuning at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/db9843994dae4489b575b0de35e6b7cf
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0


# 5. Training: Decision Tree

In [10]:
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name="Decision_Tree_RFE", nested=True):
    dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
    cv_scores_dt = cross_val_score(dt_model, X_train_rfe, y_train, cv=5, scoring='neg_mean_squared_error')
    rmse_dt = np.sqrt(-cv_scores_dt).mean()
    dt_model.fit(X_train_rfe, y_train)
    
    mlflow.log_param("model_type", "DecisionTree")
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("cv_rmse", rmse_dt)
    
    print(f"Decision Tree RMSE: {rmse_dt:.2f}")

Decision Tree RMSE: 43956.42
🏃 View run Decision_Tree_RFE at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/1db6746121d941319e3cca43f22df257
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0


# 6. Model Registration

In [11]:
import mlflow.sklearn
best_model = lasso_exp2 

model_name = "HousePrice_Best_Model"

with mlflow.start_run(run_name="Final_Model_Registration"):
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        registered_model_name=model_name
    )
    mlflow.log_metric("final_best_rmse", rmse_exp2) 
    
    print(f"Successfully registered {model_name} to DagsHub!")

2026/04/10 18:24:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'HousePrice_Best_Model' already exists. Creating a new version of this model...
2026/04/10 18:24:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: HousePrice_Best_Model, version 5
Created version '5' of model 'HousePrice_Best_Model'.


Successfully registered HousePrice_Best_Model to DagsHub!
🏃 View run Final_Model_Registration at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0/runs/0e40e98fa1064c138222d83c95eb17a0
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment1.mlflow/#/experiments/0
